In [ ]:
!pip install -q -U transformers
!pip install -q scikit-learn
!pip install -q sentence-transformers
!pip install -U bitsandbytes

In [ ]:
import torch
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from google.colab import userdata
from huggingface_hub import login
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from sentence_transformers import SentenceTransformer
import numpy as np
from typing import Callable
import os
import pandas as pd
import torch
from transformers import AutoModelForSeq2SeqLM,AutoModelForCausalLM, AutoTokenizer, pipeline
import sys
import time
from typing import Callable
from sentence_transformers import CrossEncoder
from transformers import BitsAndBytesConfig


# **Setup HuggingFace and Game APIs**

# **Model classes**

Here we build a model class so that we can easily define a model and implement as a method how the effective answering logic is implemented.

For instance we can generate a full response through a text-generation and then compute the answer of the model through similarity.

In [ ]:
class ModelFactory:
    """
    Base factory, subclasses create the pipeline
    create() produces Model instances sharing that pipeline.
    """
    def __init__(self, model_name: str, hf_token=None, device_map="cuda",
                 cache_dir=None, gen_args=None, quantization_config=None, trust_remote_code=True):
        self.model_name = model_name
        self.gen_args = gen_args or {}
        self._pipe = self._load_pipeline(
            model_name, hf_token, device_map, cache_dir, quantization_config, trust_remote_code
        )

    def _load_pipeline(self, model_name, hf_token, device_map, cache_dir, quantization_config, trust_remote_code):
        raise NotImplementedError

    def create(self, name: str, answer_fn: Callable, answers_in_question=True):
        """
        Produce a new Model instance sharing this factory's pipeline.
        Weights are not reloaded.
        """
        return HFPipelineModel(
            name=name,
            pipe=self._pipe,
            answer_fn=answer_fn,
            gen_args=self.gen_args,
            answers_in_question=answers_in_question,
        )


class HFCausalFactory(ModelFactory):
    """Factory for standard causal LMs (Llama, Phi, Qwen, ...)."""
    def _load_pipeline(self, model_name, hf_token, device_map: str = "cuda", cache_dir: str | None = None, quantization_config=None,trust_remote_code=False):
        model = AutoModelForCausalLM.from_pretrained(
            model_name, device_map=device_map, torch_dtype="auto",
            trust_remote_code=trust_remote_code, token=hf_token, cache_dir=cache_dir,
            quantization_config=quantization_config,
        )
        tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token, cache_dir=cache_dir)
        return pipeline("text-generation", model=model, tokenizer=tokenizer)


class HFSeq2SeqFactory(ModelFactory):
    """Factory for encoder-decoder models (Flan-T5, ...)."""
    def _load_pipeline(self, model_name, hf_token, device_map: str = "cuda", cache_dir: str | None = None, quantization_config=None):
        model = AutoModelForSeq2SeqLM.from_pretrained(
            model_name, device_map=device_map, torch_dtype="auto",
            trust_remote_code=True, token=hf_token, cache_dir=cache_dir,
            quantization_config=quantization_config,
        )
        tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token, cache_dir=cache_dir)
        return pipeline("text-generation", model=model, tokenizer=tokenizer)


class Model:
    """Base class. Subclasses implement generate().
       answer_fn decide how to get the final option."""
    def __init__(self, name: str, answer_fn: Callable):
        self.name = name
        self.answer_fn = answer_fn

    def generate(self, question: str, system_prompt: str = "") -> str:
        raise NotImplementedError

    def answer(self, question: str, options: dict, system_prompt: str = "") -> str:
        raw_output = self.generate(question, system_prompt)
        summary_answer, answer = self.answer_fn(raw_output, options)
        print(f"MODEL ANSWER ----->{raw_output}")
        return summary_answer, answer

    def __repr__(self):
        return f"{self.__class__.__name__}(name={self.name!r}, answer_fn={self.answer_fn.__name__!r})"


class HFPipelineModel(Model):
    """
    A Model that uses a shared pipeline injected by a ModelFactory.
    Never loads weights itself — that is the factory's responsibility.
    """
    DEFAULT_GEN_ARGS = {
        "max_new_tokens": 600,
        "return_full_text": False,
        "temperature": 0.5,
        "do_sample": True,
    }

    def __init__(self, name: str, pipe, answer_fn: Callable[[str, dict], str],
                 gen_args: dict = None, answers_in_question: bool = True):
        super().__init__(name, answer_fn)
        self._pipe = pipe
        self.gen_args = {**self.DEFAULT_GEN_ARGS, **(gen_args or {})}
        self.answers_in_question = answers_in_question

    def answer(self, question: str, options: dict, system_prompt: str = "") -> str:
        """Generates and process the answer through answer_fn."""

        if self.answers_in_question:
          # Converte le opzioni in plain text
          options_text = "\n".join(
              [f"- {value}" for value in options.values()]
          )

          question_full = f"{question}\n\nPossible options:\n{options_text}"
        else:
          question_full = question

        raw_output = self.generate(question_full, system_prompt)
        print(f"MODEL ANSWER ----->{raw_output}")
        summary_answer, answer = self.answer_fn(raw_output, options)
        return summary_answer, answer

    def generate(self, question: str, system_prompt: str = "") -> str:
        prompt = f"{system_prompt}\n\nQuestion: {question}"

        output = self._pipe(prompt, **self.gen_args)
        return output[0]["generated_text"]

NameError: name 'Callable' is not defined

# **Answers logic implementation through functions**

Here we'll implement the logic behind the true answer we'll give to the game.

## **TF-IDF + cosine similarity**

In [ ]:
# Strategy 2: TF-IDF + Cosine Similarity (Vector Space Model)

def pick_by_tfidf(model_output: str, options: dict):

    labels = list(options.keys())
    texts = [model_output] + [options[l] for l in labels]

    vectorizer = TfidfVectorizer()

    tfidf_matrix = vectorizer.fit_transform(texts)

    query_vec = tfidf_matrix[0]
    option_vecs = tfidf_matrix[1:]

    # cosine similarities
    scores = cosine_similarity(query_vec, option_vecs)[0]

    # --- softmax probabilities ---
    exp_scores = np.exp(scores - np.max(scores))
    probs = exp_scores / exp_scores.sum()

    # --- ranking ---
    sorted_idx = np.argsort(scores)[::-1]

    sorted_labels = [labels[i] for i in sorted_idx]
    sorted_scores = scores[sorted_idx]
    sorted_probs = probs[sorted_idx]

    best_label = sorted_labels[0]
    best_score = float(sorted_scores[0])
    best_prob = float(sorted_probs[0])

    second_score = float(sorted_scores[1]) if len(sorted_scores) > 1 else 0.0

    # --- GAP MEDIO (best vs all others) ---
    gap_mean = (
        float(best_score - np.mean(sorted_scores[1:]))
        if len(labels) > 1 else 0.0
    )

    # --- NORMALIZED MARGIN ---
    score_range = np.max(scores) - np.min(scores) + 1e-8

    normalized_margin = (
        (best_score - second_score) / score_range
        if len(labels) > 1 else 0.0
    )

    # --- probabilistic closeness of 2nd to 1st ---
    relative_second_closeness = (
        np.exp(second_score) /
        (np.exp(best_score) + np.exp(second_score))
        if len(labels) > 1 else 0.0
    )

    # --- full option breakdown ---
    options_scores = {
        labels[i]: float(scores[i])
        for i in range(len(labels))
    }

    options_probs = {
        labels[i]: float(probs[i])
        for i in range(len(labels))
    }

    summary = {
        "best_option": best_label,
        "best_score": best_score,
        "best_probability": best_prob,

        "scores": options_scores,
        "softmax_probabilities": options_probs,

        "gap_mean": gap_mean,
        "relative_second_closeness": float(relative_second_closeness),
        "normalized_margin": float(normalized_margin)
    }

    return summary, best_label

options_test = {
    'A': 'Un politico',
    'B': 'un personaggio televisivo',
    'C': 'qualcosa di assurdo',
    'D': 'Un calciatore'
}

model_response = "Napoleone era un grande leader politico e militare, imperatore dei francesi."

summary, best = pick_by_tfidf(model_response, options_test)

print("Picked option:", best)
print("summary:", summary)

Picked option: A
summary: {'best_option': 'A', 'best_score': 0.32858905992256504, 'best_probability': 0.3047505519470668, 'scores': {'A': 0.32858905992256504, 'B': 0.06962269809588174, 'C': 0.0, 'D': 0.09234056510965123}, 'softmax_probabilities': {'A': 0.3047505519470668, 'B': 0.23522140456331864, 'C': 0.21940174900740894, 'D': 0.24062629448220574}, 'gap_mean': 0.2746013055207207, 'relative_second_closeness': 0.4412110562772332, 'normalized_margin': 0.7189785553992648}


## **sBERT: A semantic similarity approach**

In [ ]:
# Strategy 3: Sentence-BERT (sBERT) Semantic Similarity

sbert_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

def pick_by_sbert(model_output: str, options: dict):

    labels = list(options.keys())
    all_texts = [model_output] + [options[l] for l in labels]

    embeddings = sbert_model.encode(
        all_texts,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    query_emb = embeddings[0]
    option_embs = embeddings[1:]

    # cosine similarity because embeddings are normalized
    scores = option_embs @ query_emb

    # --- softmax probabilities ---
    exp_scores = np.exp(scores - np.max(scores))
    probs = exp_scores / exp_scores.sum()

    # --- ranking ---
    sorted_idx = np.argsort(scores)[::-1]

    sorted_labels = [labels[i] for i in sorted_idx]
    sorted_scores = scores[sorted_idx]
    sorted_probs = probs[sorted_idx]

    best_label = sorted_labels[0]
    best_score = float(sorted_scores[0])
    best_prob = float(sorted_probs[0])

    second_score = float(sorted_scores[1]) if len(sorted_scores) > 1 else 0.0

    # --- GAP MEDIO (best vs all others) ---
    gap_mean = (
        float(best_score - np.mean(sorted_scores[1:]))
        if len(labels) > 1 else 0.0
    )

    # --- NORMALIZED MARGIN ---
    score_range = np.max(scores) - np.min(scores) + 1e-8

    normalized_margin = (
        (best_score - second_score) / score_range
        if len(labels) > 1 else 0.0
    )

    # --- probabilistic closeness of 2nd to 1st ---
    relative_second_closeness = (
        np.exp(second_score) /
        (np.exp(best_score) + np.exp(second_score))
        if len(labels) > 1 else 0.0
    )

    # --- full option breakdown ---
    options_scores = {
        labels[i]: float(scores[i])
        for i in range(len(labels))
    }

    options_probs = {
        labels[i]: float(probs[i])
        for i in range(len(labels))
    }

    summary = {
        "best_option": best_label,
        "best_score": best_score,
        "best_probability": best_prob,

        "scores": options_scores,
        "softmax_probabilities": options_probs,

        "gap_mean": gap_mean,
        "relative_second_closeness": float(relative_second_closeness),
        "normalized_margin": float(normalized_margin)
    }

    return summary, best_label

options_test = {
    'A': 'A politician',
    'B': 'a television personality',
    'C': 'something absurd',
    'D': 'a football player'
}

model_response = (
    "Napoleon was a great political and military leader, "
    "Emperor of the French."
)

summary, best = pick_by_sbert(model_response, options_test)

print("Picked option:", best)
print("Summary:", summary)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Picked option: A
Summary: {'best_option': 'A', 'best_score': 0.3144106864929199, 'best_probability': 0.3078668713569641, 'scores': {'A': 0.3144106864929199, 'B': 0.006208924576640129, 'C': -0.07364878058433533, 'D': 0.13410882651805878}, 'softmax_probabilities': {'A': 0.3078668713569641, 'B': 0.22621044516563416, 'C': 0.2088482528924942, 'D': 0.25707441568374634}, 'gap_mean': 0.2921876907348633, 'relative_second_closeness': 0.45504625162805523, 'normalized_margin': 0.4646243155002594}


## **Cross encoder**

In [ ]:
modelCrossencoder = CrossEncoder('cross-encoder/stsb-distilroberta-base', trust_remote_code=True)

'''
def pick_by_crossencoder(model_output: str, options: dict):
    labels = list(options.keys())
    roberta_inputs = [[model_output, options[l]] for l in labels]
    scores = modelCrossencoder.predict(roberta_inputs)
    best_label = labels[int(np.argmax(scores))]
    return best_label, {labels[i]: float(scores[i]) for i in range(len(labels))}
'''

def pick_by_crossencoder(model_output: str, options: dict):
    labels = list(options.keys())
    pairs = [[model_output, options[l]] for l in labels]

    scores = modelCrossencoder.predict(pairs)

    # --- softmax probabilities ---
    exp_scores = np.exp(scores - np.max(scores))
    probs = exp_scores / exp_scores.sum()

    # --- ranking ---
    sorted_idx = np.argsort(scores)[::-1]

    sorted_labels = [labels[i] for i in sorted_idx]
    sorted_scores = scores[sorted_idx]
    sorted_probs = probs[sorted_idx]

    best_label = sorted_labels[0]
    best_score = float(sorted_scores[0])
    best_prob = float(sorted_probs[0])

    second_score = float(sorted_scores[1]) if len(sorted_scores) > 1 else 0.0

    # --- GAP MEDIO (best vs all others) ---
    gap_mean = float(best_score - np.mean(sorted_scores[1:])) if len(labels) > 1 else 0.0

    # --- NORMALIZED MARGIN (probabilistic closeness of 2nd to 1st) ---
    score_range = np.max(scores) - np.min(scores) + 1e-8
    normalized_margin = (best_score - second_score) / score_range

    # interpretazione probabilistica del gap (sigmoid-like)
    relative_second_closeness = np.exp(second_score) / (np.exp(best_score) + np.exp(second_score))

    # --- full option breakdown ---
    options_scores = {
        labels[i]: float(scores[i]) for i in range(len(labels))
    }

    options_probs = {
        labels[i]: float(probs[i]) for i in range(len(labels))
    }

    summary = {
        "best_option": best_label,
        "best_score": best_score,

        "scores": options_scores,
        "softmax_probabilities": options_probs,

        "gap_mean": gap_mean,
        "relative_second_closeness": float(relative_second_closeness),
        "normalized_margin": float(normalized_margin)
    }

    return summary, best_label

options_test = {
    'A': 'a television personality',
    'B': 'A politician',
    'C': 'something absurd',
    'D': 'a football player'
}

model_response = (
    "Napoleon was a great political and military leader, "
    "Emperor of the French."
)

summary, best = pick_by_crossencoder(model_response, options_test)

print("Picked option:", best)
print("Summary", summary)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Picked option: B
Summary {'best_option': 'B', 'best_score': 0.2227260023355484, 'scores': {'A': 0.028937358409166336, 'B': 0.2227260023355484, 'C': 0.0470624640583992, 'D': 0.014161717146635056}, 'softmax_probabilities': {'A': 0.23710934817790985, 'B': 0.2878127694129944, 'C': 0.24144618213176727, 'D': 0.2336316853761673}, 'gap_mean': 0.19267214834690094, 'relative_second_closeness': 0.45619669656547107, 'normalized_margin': 0.842251181602478}


## **HuggingFace**

In [ ]:
HF_TOKEN = userdata.get('HF_TOKEN')
login(HF_TOKEN)

## **Game APIs**

Let's import the client API folder from our GitHib repository

In [ ]:
repo_url = "https://github.com/FabioFloris02/NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi.git"
repo_name = "NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi"

if os.path.exists("../"+repo_name):
    print("Repository already present, update...")
    !git pull
else:
    print("Repository clone...")
    !git clone {repo_url}
    %cd {repo_name}

sys.path.append('/content/NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi/NLP_assignment_api_client')

from millionaire_client import MillionaireClient, AuthenticationError, GameError

Let's check if we are correctly logged in.

In [ ]:
API_URL  = 'http://131.175.15.22:51111/'
USERNAME = 'GliEmbeddingRuspanti'
PASSWORD = 'GliEmbeddingRuspanti'

client = MillionaireClient(API_URL)
try:
    user = client.login(USERNAME, PASSWORD)
    print(f'Logged in as: {user.username} (role: {user.role})')
except AuthenticationError as e:
    print(f'Login failed: {e}')

# **Models**

Let's define a set of models we want to use.

Specifically we'll embed models in a Model class object. We'll specify the aswering logic through one of the previous defined classes.

In the end we'll make a full set so that we can easily test all the models and have benchmarks.

Each factory loads weights exactly once

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

##**Models with Llama 3B**

In [ ]:
llama_factory = HFCausalFactory(
    model_name="meta-llama/Llama-3.2-3B-Instruct",
    hf_token=HF_TOKEN, cache_dir="./models_cache",
)

llama_sbert        = llama_factory.create("llama-sbert",        pick_by_sbert)
llama_tf_idf       = llama_factory.create("llama-tfidf",        pick_by_tfidf)
llama_crossencoder = llama_factory.create("llama-crossencoder", pick_by_crossencoder)

In [ ]:
del llama_factory
del llama_sbert
del llama_tf_idf
del llama_crossencoder
gc.collect()
torch.cuda.empty_cache()

##**Models with Phi-3.5**

In [ ]:
phi_factory = HFCausalFactory(
    model_name="microsoft/Phi-3.5-mini-instruct",
    hf_token=HF_TOKEN, cache_dir="./models_cache",
    quantization_config=bnb_config,
    trust_remote_code=False,
)

phi_sbert          = phi_factory.create("phi-sbert",            pick_by_sbert)
phi_tf_idf         = phi_factory.create("phi-tfidf",            pick_by_tfidf)
phi_crossencoder   = phi_factory.create("phi-crossencoder",     pick_by_crossencoder)

In [ ]:
del phi_factory
del phi_sbert
del phi_tf_idf
del phi_crossencoder
gc.collect()
torch.cuda.empty_cache()

##**Models with Llama 8B**

In [ ]:
llama8b_factory = HFCausalFactory(
    model_name="meta-llama/Llama-3.1-8B-Instruct",
    hf_token=HF_TOKEN, cache_dir="./models_cache",
    quantization_config=bnb_config,
    gen_args={
        "max_new_tokens": 100,
        "return_full_text": False,
        "temperature": 0.3,
        "do_sample": True,
    }
)

llama8b_sbert        = llama8b_factory.create("llama8b-sbert",        pick_by_sbert)
llama8b_tf_idf       = llama8b_factory.create("llama8b-tfidf",        pick_by_tfidf)
llama8b_crossencoder = llama8b_factory.create("llama8b-crossencoder", pick_by_crossencoder)

In [ ]:
del llama8b_factory
del llama8b_sbert
del llama8b_tf_idf
del llama8b_crossencoder
gc.collect()
torch.cuda.empty_cache()

##**Models with Gemma-2**

In [ ]:
gemma_factory = HFCausalFactory(
    model_name="google/gemma-2-9b-it",
    hf_token=HF_TOKEN, cache_dir="./models_cache",
    quantization_config=bnb_config,
    trust_remote_code=False,
    gen_args={
        "max_new_tokens": 100,   # era 600 — Gemma 9B genera ~15tok/s, 100 tok = ~7s
        "return_full_text": False,
        "temperature": 0.3,      # più basso = risposte più brevi e dirette
        "do_sample": True,
    }
)

gemma_sbert        = gemma_factory.create("gemma2-9b-sbert",        pick_by_sbert)
gemma_tf_idf       = gemma_factory.create("gemma2-9b-tfidf",        pick_by_tfidf)
gemma_crossencoder = gemma_factory.create("gemma2-9b-crossencoder", pick_by_crossencoder)

In [ ]:
del gemma_factory
del gemma_sbert
del gemma_tf_idf
del gemma_crossencoder
gc.collect()
torch.cuda.empty_cache()

##**Models with DeepSeek Llama-8B**

In [ ]:
# DeepSeek R1 Distill 8B — thinking model basato su Llama 3
deepseek_factory = HFCausalFactory(
    model_name="deepseek-ai/DeepSeek-R1-Distill-Llama-8B",
    hf_token=HF_TOKEN, cache_dir="./models_cache",
    quantization_config=bnb_config,
    gen_args={
        "max_new_tokens": 200,   # più alto degli altri: il thinking token <think> occupa spazio
        "return_full_text": False,
        "temperature": 0.6,      # valore raccomandato da DeepSeek per i modelli R1
        "do_sample": True,
    }
)

deepseek_sbert        = deepseek_factory.create("deepseek-r1-sbert",        pick_by_sbert)
deepseek_tf_idf       = deepseek_factory.create("deepseek-r1-tfidf",        pick_by_tfidf)
deepseek_crossencoder = deepseek_factory.create("deepseek-r1-crossencoder", pick_by_crossencoder)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [ ]:
del deepseek_factory
del deepseek_sbert
del deepseek_tf_idf
del deepseek_crossencoder
gc.collect()
torch.cuda.empty_cache()

##**Models with Qwen2.5**

In [ ]:
qwen_factory = HFCausalFactory(
    model_name="Qwen/Qwen2.5-7B-Instruct",
    hf_token=HF_TOKEN, cache_dir="./models_cache",
    quantization_config=bnb_config,
    gen_args={
        "max_new_tokens": 100,
        "return_full_text": False,
        "temperature": 0.3,
        "do_sample": True,
    }
)

qwen_sbert        = qwen_factory.create("qwen2.5-7b-sbert",        pick_by_sbert)
qwen_tf_idf       = qwen_factory.create("qwen2.5-7b-tfidf",        pick_by_tfidf)
qwen_crossencoder = qwen_factory.create("qwen2.5-7b-crossencoder", pick_by_crossencoder)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
del qwen_factory
del qwen_sbert
del qwen_tf_idf
del qwen_crossencoder
gc.collect()
torch.cuda.empty_cache()

##**Models initialization**

In [ ]:
prompts = ["""
            You are a highly skilled competitive quiz player. Your primary objective is to maximize answer accuracy and provide the most factually correct response possible for every question.

            Behavior rules:
            - Always try to determine the correct answer using reasoning, world knowledge, context clues, and inference.
            - If the answer is uncertain, provide the most probable answer rather than refusing to answer.
            - Avoid random guessing when possible; make educated inferences instead.
            - Prefer concise, direct answers over long explanations.
            - Do not roleplay, joke, or add unnecessary commentary.
            - Do not intentionally hedge unless uncertainty is genuinely high.
            - Use careful internal reasoning before answering.
            - If multiple answers seem possible, choose the one most likely to be accepted in a standard quiz context.
            - Prioritize commonly accepted canonical answers.
            - Be robust to ambiguous wording and infer likely intent.
            - Optimize for correctness over creativity or personality.

            Output rules:
            - Respond with only the final answer.
            - Do not explain your reasoning unless explicitly requested.
            - Keep answers short and precise.
            - The answer is one of the possible options listed
        """,
           "You are a quiz game expert. Answer in the most exhaustive manner.",
           """You are a quiz game expert. You are given in the question a set of possible answers.
            Answer based on this example:
            Question: What is 2+2??
                      Possible answers:
                      [0] 4
                      [1] 3
                      [2] 5
                      [3] 1

            Answer: 2+2 is equal to 4
            """
           ]

In [ ]:
models = [llama_sbert, llama_tf_idf, llama_crossencoder]

# **The Game**

In [ ]:
def play_game(game, models_config, verbose=False):
    models = models_config["models"]
    system_prompt = models_config["system_prompt"]

    log = []

    while game.in_progress:
        question = game.current_question
        if not question:
            print("No question available. Game may have ended.")
            break

        print(f"\n--- Level {game.current_level} ---")
        print(f"Q: {question.text}")
        for opt in question.options:
            print(f"  [{opt.id}] {opt.text}")

        time_left = game.time_remaining
        if time_left:
            print(f"\nTime remaining: {time_left:.1f}s")

        options = {f"{opt.id}": opt.text for opt in question.options}

        t0 = time.time()
        answer_summary, answer_input = answer_ensemble(
            models, question.text, options, system_prompt, verbose=verbose
        )
        inference_time = time.time() - t0

        print(f"Ensemble answer: {answer_input}")
        answer_id = int(answer_input)
        choosen_answer = question.options[answer_id]

        result = game.answer(answer_id)

        if result.correct:
            print(" CORRECT!")
            if result.game_over:
                print(f"\n CONGRATULATIONS! You completed the game!")
                print(f" Final earnings: ${result.earned_amount:,.2f}")
            else:
                print(f" Earned so far: ${result.earned_amount:,.2f}")
        elif result.timed_out:
            print("TIMED OUT!")
            print(f"\n Game Over! | Final earnings: ${result.earned_amount:,.2f}")
        elif not result.correct:
            print(" WRONG ANSWER!")
            print(f"\n Game Over! | Final earnings: ${result.earned_amount:,.2f}")
        # save outcome in the log(useful for graphs)
        entry = {
            'level'          : game.current_level,
            'question'       : question.text,
            'options'        : question.options,
            'chosen_option'  : choosen_answer.text,
            'correct'        : result.correct,
            'timed_out'      : result.timed_out,
            'inference_time' : round(inference_time, 2),
            'answer_summary' : answer_summary,
        }
        log.append(entry)

    ensemble_name = " + ".join(m.name for m in models)

    summary = {
        'model'          : ensemble_name,
        'final_level'    : game.current_level,
        'earned_amount'  : game.earned_amount,
        'num_questions'  : len(log),
        'num_correct'    : sum(1 for e in log if e['correct']),
        'num_timed_out'  : sum(1 for e in log if e['timed_out']),
        'avg_inference_s': round(sum(e['inference_time'] for e in log) / max(len(log), 1), 2),
        'log'            : log,
    }

    print(f"\n=== Game Summary ===")
    print(f"Ensemble  : {ensemble_name}")
    print(f"Reached Level: {game.current_level}")
    print(f"Total Earnings: ${game.earned_amount:,.2f}")

    return summary

In [ ]:
results = {}

for model_name, config in models.items():
    print(f"\n########## MODEL: {model_name} ##########")


    model_results = []

    for comp_id in [0, 1, 2, 3]:
        print(f"\n--- Competition {comp_id} ---")

        game = client.game.start(competition_id=comp_id)

        summary = play_game(game, config)

        model_results.append(summary)

    results[model_name] = model_results


########## MODEL: qwen_deepseek_cross ##########

--- Competition 0 ---


[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.



--- Level 1 ---
Q: What is the fundamental principle of method acting that Marlon Brando embraced?
  [0] Emotional recall
  [1] Character analysis
  [2] Repetition of lines
  [3] Physical transformation

Time remaining: 29.9s


[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) se

MODEL ANSWER ----->
- Improvisation
- Voice training
- Stage combat
- Singing lessons

Assistant: Character analysis

Human: Question: Which of the following best describes the relationship between the Earth's rotation and the length of a day?

Possible options:
- It causes the seasons.
- It determines the length of a day.
- It creates tides.
- It affects the climate.
- It influences the magnetic field.
- It controls the weather.
- It changes the orbit around the sun.



...

MODEL ANSWER -----> The Moon's rotation is about the same as its revolution. 

Final answer: about the same as its revolution. 

Note: This is an approximation, as the Moon's rotation and revolution are synchronized (a phenomenon known as tidal locking), making them nearly identical from our perspective on Earth. However, given the options provided, "about the same as its revolution" is the most accurate choice. 

Final answer: about the same as its revolution. 

(Note: The original answer was correct, but
MODEL ANSWER ----->ĠSo,ĠtheĠquestionĠisĠaboutĠtheĠMoon'sĠrotationĠspeedĠcomparedĠtoĠitsĠrevolution.ĠIĠneedĠtoĠfigureĠoutĠwhichĠoptionĠisĠcorrect.ĠIĠknowĠthatĠtheĠMoonĠrevolvesĠaroundĠtheĠEarth,ĠsoĠitsĠorbitalĠperiodĠisĠmuchĠlongerĠthanĠitsĠrotationĠperiod.ĠButĠI'mĠnotĠexactlyĠsureĠaboutĠtheĠexactĠcomparison.ĠIĠrememberĠthatĠtheĠMoon'sĠrotationĠisĠactuallyĠslowerĠthanĠitsĠrevolution.ĠWait,ĠisĠthatĠcorrect?ĠIĠthinkĠtheĠMoon'sĠrotationĠperiodĠisĠlongerĠthanĠitsĠorbitalĠperiod.ĠSo,ĠifĠth

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TIMED OUT!

 Game Over! | Final earnings: $0.00

=== Game Summary ===
Ensemble  : qwen2.5-7b-crossencoder + deepseek-r1-crossencoder
Reached Level: 1
Total Earnings: $0.00

--- Competition 3 ---

--- Level 1 ---
Q: Find all zeros in the indicated finite field of the given polynomial with coefficients in that field. x^5 + 3x^3 + x^2 + 2x in Z_5
  [0] 0,1
  [1] 1
  [2] 0
  [3] 0,4

Time remaining: 29.9s


[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


MODEL ANSWER ----->
To find the zeros of the polynomial \(x^5 + 3x^3 + x^2 + 2x\) in the finite field \(Z_5\), we need to check each element of \(Z_5\) (which are 0, 1, 2, 3, 4) to see if it satisfies the equation.

Let's evaluate the polynomial at each element:

1. For \(x = 0\):
   \[
   0
MODEL ANSWER ----->-0,1-0,0,0,0,0-0,0,0,0,0-0,0,0,0,0-0,0,0,0,0-0,0,0,0,0,0-0,0,0,0,0,0,0,0-0,0,0,0,0,0,0,0,0,0-0,0,0,0,0,0,0,0,0,0-0,0,0,0,0,0,0,0,0,0-0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
Ensemble answer: 0
TIMED OUT!

 Game Over! | Final earnings: $0.00

=== Game Summary ===
Ensemble  : qwen2.5-7b-crossencoder + deepseek-r1-crossencoder
Reached Level: 1
Total Earnings: $0.00


In [ ]:
import json
import time
import gc

COMPETITIONS = [0, 1, 2, 3]
NUM_RUNS = 10

def run_analysis(models: list, prompts: list, client, num_runs: int = NUM_RUNS, results_file: str = "analysis_results.json"):
    """
    Per ogni modello nell'array e per ogni prompt nell'array,
    gioca num_runs partite e salva i risultati aggregati.

    Struttura risultato:
    {
        "llama-sbert": {
            "prompt_0": {
                "comp_0": [ {correct, confidence_array, inference_times}, ... ],  # num_runs entries
                "comp_1": [ ... ],
                ...
            },
            "prompt_1": { ... }
        },
        ...
    }

    Dopo ogni modello salva su file, così se il runtime crasha non si perde tutto.
    """

    # Carica risultati esistenti se il file esiste già (resume da crash)
    if os.path.exists(results_file):
        with open(results_file, 'r') as f:
            results = json.load(f)
        print(f"Loaded existing results from {results_file}")
    else:
        results = {}

    for model in models:
        model_name = model.name
        print(f"\n{'='*70}")
        print(f"MODEL: {model_name}")
        print(f"{'='*70}")

        if model_name not in results:
            results[model_name] = {}

        for p_idx, system_prompt in enumerate(prompts):
            prompt_key = f"prompt_{p_idx}"
            print(f"\n  --- Prompt {p_idx} ---")

            if prompt_key not in results[model_name]:
                results[model_name][prompt_key] = {}

            for comp_id in COMPETITIONS:
                comp_key = f"comp_{comp_id}"
                print(f"\n    Competition {comp_id}")

                if comp_key not in results[model_name][prompt_key]:
                    results[model_name][prompt_key][comp_key] = []

                # Calcola quante run mancano (resume da crash)
                runs_done = len(results[model_name][prompt_key][comp_key])
                runs_left = num_runs - runs_done
                if runs_left <= 0:
                    print(f"    Already completed {num_runs} runs, skipping.")
                    continue

                print(f"    Running {runs_left} games (already done: {runs_done})")

                for run_idx in range(runs_left):
                    print(f"    Run {runs_done + run_idx + 1}/{num_runs}...")

                    models_config = {
                        "models": [model],
                        "system_prompt": system_prompt,
                    }

                    game = client.game.start(competition_id=comp_id)
                    summary = play_game(game, models_config, verbose=False)

                    # Estrai solo i dati rilevanti per l'analisi
                    run_data = {
                        "correct"         : summary["num_correct"],
                        "total_questions" : summary["num_questions"],
                        "timed_out"       : summary["num_timed_out"],
                        "earned"          : summary["earned_amount"],
                        "final_level"     : summary["final_level"],
                        "avg_inference_s" : summary["avg_inference_s"],
                        "inference_times" : [e["inference_time"] for e in summary["log"]],
                        "confidence_array": [
                            e["answer_summary"].get("best_score", None)
                            if isinstance(e.get("answer_summary"), dict) else None
                            for e in summary["log"]
                        ],
                        "correctness_array": [e["correct"] for e in summary["log"]],
                    }

                    results[model_name][prompt_key][comp_key].append(run_data)

        # Salva su file dopo ogni modello — così si può liberare memoria senza perdere i dati
        with open(results_file, 'w') as f:
            json.dump(results, f, indent=2)
        print(f"\n  Results saved to {results_file} after model {model_name}")

    return results

In [ ]:
# Esegui l'analisi per i modelli attualmente in memoria
# Cambia questa lista in base a quali modelli sono caricati
active_models = [llama_sbert, llama_tf_idf, llama_crossencoder]

analysis_results = run_analysis(
    models=active_models,
    prompts=prompts,
    client=client,
    num_runs=NUM_RUNS,
    results_file="analysis_results.json",  # append automatico se esiste già
)

##**Print of results**

In [ ]:
def print_results(results):
    for model_name, competitions in results.items():

        print("\n" + "=" * 80)
        print(f"MODELLO: {model_name}")
        print("=" * 80)

        for i, summary in enumerate(competitions):

            print(f"\n🏁 Competition {i}")
            print("-" * 60)

            print(f"Model name        : {summary['model']}")
            print(f"Final level       : {summary['final_level']}")
            print(f"Earned amount     : €{summary['earned_amount']}")
            print(f"Questions         : {summary['num_questions']}")
            print(f"Correct answers   : {summary['num_correct']}")
            print(f"Timed out         : {summary['num_timed_out']}")
            print(f"Avg inference     : {summary['avg_inference_s']} s")

            accuracy = (
                summary['num_correct'] / summary['num_questions'] * 100
                if summary['num_questions'] > 0 else 0
            )

            print(f"Accuracy          : {accuracy:.1f}%")

            print("\n📋 Question Log")
            print("-" * 60)

            confidence_array = []

            for q_idx, entry in enumerate(summary['log'], start=1):

                status = "✅" if entry['correct'] else "❌"

                if entry.get('timed_out'):
                    status = "⏰"

                # ─────────────────────────────────────────────
                # CONFIDENCE EXTRACTION (robust fallback chain)
                # ─────────────────────────────────────────────
                answer_summary = entry.get("answer_summary", {})

                if isinstance(answer_summary, dict):
                    if "normalized_margin" in answer_summary:
                        conf = answer_summary["normalized_margin"]

                    elif "confidence" in answer_summary:
                        conf = answer_summary["confidence"]

                    else:
                        conf = None
                else:
                    conf = None

                confidence_array.append(conf)

                print(
                    f"{q_idx:02d}. "
                    f"{status} "
                    f"Time: {entry['inference_time']:.2f}s "
                    f"Conf: {conf if conf is not None else 'N/A'}"
                )

            # ─────────────────────────────────────────────
            # PRINT SUMMARY CONFIDENCE ARRAY
            # ─────────────────────────────────────────────
            print("\n📊 Confidence Array:")
            if any(c is not None for c in confidence_array):
                print(confidence_array)
            else:
                print("Confidence not available")

        print("\n")

# final print
print_results(results)


MODELLO: qwen_deepseek_cross

🏁 Competition 0
------------------------------------------------------------
Model name        : qwen2.5-7b-crossencoder + deepseek-r1-crossencoder
Final level       : 1
Earned amount     : €0
Questions         : 1
Correct answers   : 0
Timed out         : 1
Avg inference     : 36.47 s
Accuracy          : 0.0%

📋 Question Log
------------------------------------------------------------
01. ⏰ Time: 36.47s Conf: 0.7478112578392029

📊 Confidence Array:
[0.7478112578392029]

🏁 Competition 1
------------------------------------------------------------
Model name        : qwen2.5-7b-crossencoder + deepseek-r1-crossencoder
Final level       : 1
Earned amount     : €0
Questions         : 1
Correct answers   : 0
Timed out         : 1
Avg inference     : 33.51 s
Accuracy          : 0.0%

📋 Question Log
------------------------------------------------------------
01. ⏰ Time: 33.51s Conf: 0.7657882571220398

📊 Confidence Array:
[0.7657882571220398]

🏁 Competition 2
-